In [1]:
import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset

from pathlib import Path

In [2]:
base_path = Path("..")

processed_path = base_path / "data" / "processed"
wet_path = processed_path / "wet"
data_split_path = processed_path / "data_split"

train_path = data_split_path / "train.csv"
val_path = data_split_path / "val.csv"
test_path = data_split_path / "test.csv"

In [3]:
train_df = pd.read_csv(train_path)

train_df.head()

,source_id,wet_path,room_size,wet_level,rate_hz,depth,room_size_norm,wet_level_norm,rate_hz_norm,depth_norm
0,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v0.wav,0.374540,0.570429,3.793973,0.598658,0.374540,0.950714,0.731994,0.598658
1,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v1.wav,0.156019,0.093597,0.761376,0.866176,0.156019,0.155995,0.058084,0.866176
2,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v2.wav,0.601115,0.424844,0.592630,0.969910,0.601115,0.708073,0.020584,0.969910
3,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v3.wav,0.832443,0.127403,1.318212,0.183405,0.832443,0.212339,0.181825,0.183405
4,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v4.wav,0.304242,0.314854,2.443753,0.291229,0.304242,0.524756,0.431945,0.291229


In [4]:
audio_path = train_df.loc[0, "wet_path"]

y, sr = librosa.load(
    audio_path,
    sr=None
)

print(audio_path)
print(y.shape)
print(sr)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


..\data\processed\wet\Bridge_1-0_v0.wav
(240000,)
48000


#### Матрица частот и временных отрезков

In [5]:
mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_fft=2048,
    hop_length=512,
    n_mels=128
)

mel_spec.shape

(128, 469)

In [6]:
mel_db = librosa.power_to_db(
    mel_spec,
    ref=np.max
)

print(mel_db.shape)
print(mel_db.min())
print(mel_db.max())

(128, 469)
-80.0
0.0


#### Нормирование

In [7]:
mel_norm = (mel_db + 80) / 80
print(mel_norm.min())
print(mel_norm.max())
print(mel_norm.shape)

0.0
1.0
(128, 469)


#### Тензоры

In [8]:
mel_tensor = torch.from_numpy(mel_norm).unsqueeze(0)

print(mel_tensor.shape)
mel_tensor.dtype

torch.Size([1, 128, 469])


torch.float32

#### Класс для перевода в тензоры

In [18]:
class GuitarFXDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        audio_path = row["wet_path"]

        y, sr = librosa.load(
            audio_path,
            sr=None
        )

        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=2048,
            hop_length=512,
            n_mels=128
        )

        mel_db = librosa.power_to_db(
            mel_spec,
            ref=np.max
        )

        mel_norm = (mel_db + 80) / 80

        mel_tensor = torch.from_numpy(mel_norm).unsqueeze(0)

        targets_cols = [
            "room_size_norm",
            "wet_level_norm",
            "rate_hz_norm",
            "depth_norm"
        ]   

        target = row[targets_cols].to_numpy(dtype=np.float32)
        target_tensor = torch.from_numpy(target)


        return mel_tensor, target_tensor

In [20]:
train_dataset = GuitarFXDataset(train_df)

x, y = train_dataset[0]

print(x.shape)
print(x.dtype)

print(y.shape)
print(y.dtype)
print(y)




torch.Size([1, 128, 469])
torch.float32
torch.Size([4])
torch.float32
tensor([0.3745, 0.9507, 0.7320, 0.5987])
